# Layer-wise BERT Probing on SentEval (3 Tasks)

This notebook samples **50,000 examples per task**, balances classes, makes a **stratified 90:10 train/test split**, extracts sentence representations from each of the **12 encoder layers of `bert-base-uncased`**, and trains one multinomial/binary **Logistic Regression** classifier per layer.

Tasks:
- `sentence_length.txt` — SentLen (6 classes)
- `bigram_shift.txt` — BShift (2 classes)
- `subj_number.txt` — SubjNum (2 classes)

**Important:** this notebook downloads the official SentEval files and pretrained BERT weights at runtime, so it needs internet access on the machine/Colab. The complete experiment is compute-intensive; a GPU is strongly recommended. It saves representations and result tables to disk so a session can be resumed. No benchmark results are fabricated in this notebook; run the experiment to produce your actual scores.

Reference: Conneau et al., *What you can cram into a single $&!#* vector: Probing sentence embeddings for linguistic properties* (ACL 2018). SentEval task definitions: https://github.com/facebookresearch/SentEval/tree/main/data/probing


In [ ]:
# Install dependencies (run once)
%pip -q install transformers scikit-learn pandas matplotlib tqdm requests

In [ ]:
import os, random, time, gc, json, requests
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
N_SAMPLES = 50_000
TRAIN_SIZE = 0.90
MODEL_NAME = "bert-base-uncased"   # BERT-base: 12 Transformer encoder layers
MAX_LENGTH = 64
BATCH_SIZE = 32 if torch.cuda.is_available() else 8
OUTPUT_DIR = Path("probing_outputs")
DATA_DIR = Path("senteval_probing_data")
FEATURE_DIR = OUTPUT_DIR / "features"
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
FEATURE_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Batch size:", BATCH_SIZE)
print("Model:", MODEL_NAME)


## 1. Download and inspect the official SentEval data

SentEval rows are tab-separated: partition, class label, and sentence (the last field). This parser handles extra tab-separated fields by taking the last field as the sentence.

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/facebookresearch/SentEval/main/data/probing"
TASK_FILES = {
    "SentLen": "sentence_length.txt",
    "BShift": "bigram_shift.txt",
    "SubjNum": "subj_number.txt",
}

def download_task(filename):
    url = f"{BASE_URL}/{filename}"
    dest = DATA_DIR / filename
    if not dest.exists() or dest.stat().st_size == 0:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        dest.write_bytes(response.content)
    return dest

def read_senteval_file(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.rstrip("\\n\\r")
            if not line:
                continue
            fields = line.split("\\t")
            if len(fields) < 3:
                continue
            # Official format: partition, label, ..., sentence. Sentence is last field.
            partition, label, sentence = fields[0], fields[1], fields[-1]
            if sentence.strip():
                rows.append((partition, label, sentence.strip()))
    return pd.DataFrame(rows, columns=["partition", "label", "sentence"])

raw_dfs = {}
for task, filename in TASK_FILES.items():
    file_path = download_task(filename)
    df = read_senteval_file(file_path)
    raw_dfs[task] = df
    print(f"\\n{task}: {len(df):,} rows")
    display(df.head(3))
    print("Labels:", df["label"].value_counts().to_dict())


## 2. Create balanced 50,000-example datasets and stratified 90:10 splits

The target is the same number of examples from each class. For each task, the code uses the minimum class count to determine whether 50,000 balanced samples are possible. If a task has fewer than `50,000 / number_of_classes` examples in any class, it raises an informative error rather than silently producing an imbalanced dataset. Sampling is performed across all partitions, as requested, then the split is stratified.

In [ ]:
def make_balanced_splits(df, task_name, n_samples=N_SAMPLES, train_size=TRAIN_SIZE):
    # Avoid duplicate rows, then balance by the rarest class.
    df = df[["label", "sentence"]].dropna().drop_duplicates().copy()
    counts = df["label"].value_counts()
    classes = sorted(counts.index.tolist())
    n_classes = len(classes)
    per_class = n_samples // n_classes
    if n_samples % n_classes != 0:
        raise ValueError(f"{task_name}: {n_samples=} must be divisible by {n_classes=} for exact balance.")
    too_small = {c: int(counts[c]) for c in classes if counts[c] < per_class}
    if too_small:
        raise ValueError(
            f"{task_name}: cannot sample {n_samples:,} balanced unique rows. "
            f"Need {per_class:,} per class, but these classes have fewer: {too_small}. "
            "Inspect label counts or lower N_SAMPLES."
        )
    balanced = pd.concat([
        df[df.label == c].sample(n=per_class, random_state=SEED)
        for c in classes
    ], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

    train_df, test_df = train_test_split(
        balanced, train_size=train_size, random_state=SEED,
        stratify=balanced["label"]
    )
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print(f"\\n{task_name}: total={len(balanced):,}, train={len(train_df):,}, test={len(test_df):,}")
    print("All-data class counts:", balanced.label.value_counts().sort_index().to_dict())
    print("Train class counts:    ", train_df.label.value_counts().sort_index().to_dict())
    print("Test class counts:     ", test_df.label.value_counts().sort_index().to_dict())
    assert len(balanced) == n_samples
    return train_df, test_df

splits = {}
for task, df in raw_dfs.items():
    splits[task] = make_balanced_splits(df, task)
    splits[task][0].to_csv(OUTPUT_DIR / f"{task}_train.csv", index=False)
    splits[task][1].to_csv(OUTPUT_DIR / f"{task}_test.csv", index=False)


## 3. Load BERT and extract representations from all 12 encoder layers

For each sentence, this notebook uses **mean pooling over non-padding, non-special tokens** from each encoder layer. It excludes the embedding output and uses the outputs of Transformer layers 1–12, yielding exactly **12 classifiers per task**. This is a probing setup: BERT is frozen; only Logistic Regression is trained.

Feature arrays are stored as float32 NumPy memmaps on disk, one file per task/split/layer. This avoids keeping all 12 layers for all examples in RAM at once.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)
bert.to(device)
bert.eval()
for p in bert.parameters():
    p.requires_grad = False

N_LAYERS = bert.config.num_hidden_layers
HIDDEN_SIZE = bert.config.hidden_size
print("Encoder layers:", N_LAYERS, "| hidden size:", HIDDEN_SIZE)
assert N_LAYERS == 12, "This assignment expects BERT-base with 12 encoder layers."


In [ ]:
@torch.inference_mode()
def extract_layer_features(sentences, output_prefix, batch_size=BATCH_SIZE):
    """Extract mean-pooled token representations for layers 1..12 into .npy memmaps."""
    n = len(sentences)
    maps = []
    paths = []
    for layer_idx in range(1, N_LAYERS + 1):
        path = Path(f"{output_prefix}_layer{layer_idx:02d}.npy")
        paths.append(path)
        maps.append(np.lib.format.open_memmap(
            path, mode="w+", dtype=np.float32, shape=(n, HIDDEN_SIZE)
        ))

    for start in tqdm(range(0, n, batch_size), desc=f"Encoding {Path(output_prefix).name}"):
        batch_sentences = sentences[start:start + batch_size]
        encoded = tokenizer(
            batch_sentences, padding=True, truncation=True, max_length=MAX_LENGTH,
            return_tensors="pt", return_special_tokens_mask=True
        )
        special_mask = encoded.pop("special_tokens_mask").to(device)
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        outputs = bert(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        # Keep ordinary tokens only; fallback to attention mask if a sentence has no ordinary tokens.
        token_mask = attention_mask.bool() & (~special_mask.bool())
        empty_rows = token_mask.sum(dim=1) == 0
        if empty_rows.any():
            token_mask[empty_rows] = attention_mask[empty_rows].bool()

        denom = token_mask.sum(dim=1, keepdim=True).clamp(min=1).unsqueeze(-1)
        for layer_idx in range(1, N_LAYERS + 1):
            h = outputs.hidden_states[layer_idx]
            pooled = (h * token_mask.unsqueeze(-1)).sum(dim=1) / denom.squeeze(1)
            maps[layer_idx - 1][start:start + len(batch_sentences)] = pooled.detach().cpu().numpy().astype(np.float32)

        del encoded, outputs, input_ids, attention_mask, special_mask, token_mask
        if device.type == "cuda" and start % (batch_size * 20) == 0:
            torch.cuda.empty_cache()

    for m in maps:
        m.flush()
    del maps
    return paths

def features_exist(prefix, n_layers=N_LAYERS):
    return all(Path(f"{prefix}_layer{i:02d}.npy").exists() for i in range(1, n_layers + 1))


## 4. Train 12 Logistic Regression models per task and evaluate on the held-out test set

The test set is not used for fitting. Logistic Regression uses standardized features (scaling is fitted only on the training set) and class-balanced sampling makes accuracy directly interpretable. `max_iter` is increased to help convergence.

In [ ]:
all_results = []

for task, (train_df, test_df) in splits.items():
    print(f"\\n{'='*18} {task} {'='*18}")
    train_prefix = str(FEATURE_DIR / f"{task}_train")
    test_prefix = str(FEATURE_DIR / f"{task}_test")

    if not features_exist(train_prefix) or not features_exist(test_prefix):
        extract_layer_features(train_df.sentence.tolist(), train_prefix)
        extract_layer_features(test_df.sentence.tolist(), test_prefix)
    else:
        print("Found cached representations; skipping extraction.")

    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(train_df.label.astype(str))
    y_test = label_encoder.transform(test_df.label.astype(str))

    for layer_idx in range(1, N_LAYERS + 1):
        X_train = np.load(f"{train_prefix}_layer{layer_idx:02d}.npy", mmap_mode="r")
        X_test = np.load(f"{test_prefix}_layer{layer_idx:02d}.npy", mmap_mode="r")

        # Fit scaler only on training data; use float32 to limit memory use.
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler(copy=True)
        X_train_scaled = scaler.fit_transform(np.asarray(X_train, dtype=np.float32))
        X_test_scaled = scaler.transform(np.asarray(X_test, dtype=np.float32))

        clf = LogisticRegression(
            C=1.0, solver="lbfgs", max_iter=1000, random_state=SEED,
            n_jobs=None
        )
        t0 = time.time()
        clf.fit(X_train_scaled, y_train)
        pred = clf.predict(X_test_scaled)
        elapsed = time.time() - t0

        row = {
            "task": task,
            "layer": layer_idx,
            "accuracy": accuracy_score(y_test, pred),
            "macro_f1": f1_score(y_test, pred, average="macro"),
            "weighted_f1": f1_score(y_test, pred, average="weighted"),
            "fit_eval_seconds": round(elapsed, 2),
            "n_train": len(y_train),
            "n_test": len(y_test),
            "n_classes": len(label_encoder.classes_),
        }
        all_results.append(row)
        print(f"Layer {layer_idx:02d}: accuracy={row['accuracy']:.4f}, "
              f"macro-F1={row['macro_f1']:.4f}, time={elapsed:.1f}s")

        # Save after each layer so partial results survive an interrupted session.
        pd.DataFrame(all_results).to_csv(OUTPUT_DIR / "layerwise_results_partial.csv", index=False)
        del X_train, X_test, X_train_scaled, X_test_scaled, scaler, clf, pred
        gc.collect()

results = pd.DataFrame(all_results).sort_values(["task", "layer"]).reset_index(drop=True)
results.to_csv(OUTPUT_DIR / "layerwise_results.csv", index=False)
display(results)


## 5. Compare layer-wise results

The plot compares held-out accuracy across the 12 BERT encoder layers for each task. Interpret the trend rather than expecting one universal best layer: SentLen, BShift, and SubjNum probe different kinds of information.

In [ ]:
if "results" not in globals() or results.empty:
    results_path = OUTPUT_DIR / "layerwise_results.csv"
    if results_path.exists():
        results = pd.read_csv(results_path)
    else:
        raise RuntimeError("Run the training cell first to generate results.")

fig, ax = plt.subplots(figsize=(10, 5))
for task, group in results.groupby("task"):
    group = group.sort_values("layer")
    ax.plot(group["layer"], group["accuracy"], marker="o", label=task)
ax.set_xticks(range(1, 13))
ax.set_xlabel("BERT encoder layer")
ax.set_ylabel("Test accuracy")
ax.set_title("Layer-wise Logistic Regression Probing Accuracy")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "layerwise_accuracy.png", dpi=160)
plt.show()

display(results.pivot(index="layer", columns="task", values="accuracy").round(4))


In [ ]:
# Optional: detailed classification report for the best layer of each task.
for task in results.task.unique():
    best_row = results[results.task == task].sort_values("accuracy", ascending=False).iloc[0]
    layer_idx = int(best_row.layer)
    train_df, test_df = splits[task]
    train_prefix = str(FEATURE_DIR / f"{task}_train")
    test_prefix = str(FEATURE_DIR / f"{task}_test")
    enc = LabelEncoder()
    y_train = enc.fit_transform(train_df.label.astype(str))
    y_test = enc.transform(test_df.label.astype(str))
    from sklearn.preprocessing import StandardScaler
    Xtr = np.asarray(np.load(f"{train_prefix}_layer{layer_idx:02d}.npy", mmap_mode="r"), dtype=np.float32)
    Xte = np.asarray(np.load(f"{test_prefix}_layer{layer_idx:02d}.npy", mmap_mode="r"), dtype=np.float32)
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xte = scaler.transform(Xte)
    clf = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=SEED)
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xte)
    print(f"\\n{task} — best layer {layer_idx}, accuracy {accuracy_score(y_test, pred):.4f}")
    print(classification_report(y_test, pred, target_names=enc.classes_, digits=4))


## Notes for the report

- Report the exact sampling, balancing, split ratio, pooling method, BERT checkpoint, and Logistic Regression settings.
- The requested 50,000 examples per task means 45,000 train and 5,000 test examples.
- This implementation uses only encoder layers 1–12 (not the embedding output), so it trains 36 classifiers total.
- Accuracy is the main comparison metric; macro-F1 is included as a useful secondary metric.
- Results can vary with the random seed, library versions, checkpoint revision, pooling choice, and regularization.
- SentEval's original probing benchmarks used their own fixed partitions and protocols. This assignment's random balanced 50k sample and 90:10 split are a different protocol, so compare trends cautiously rather than treating paper scores as directly identical.
- Cite: Conneau et al. (2018), *What you can cram into a single $&!#* vector: Probing sentence embeddings for linguistic properties*, ACL. Dataset documentation: https://github.com/facebookresearch/SentEval/blob/main/data/probing/README.md
